In [2]:
##Primaire netwerkinterface
##Mijn script kiest de netwerkkaart die aan staat, een geldig IPv4-adres heeft (niet 127.x of 169.254.x) en geen virtuele adapter is (VirtualBox, VPN, Bluetooth). Als er dan nog meerdere overblijven, kies ik die met het meeste verkeer. Virtuele adapters hebben ook een IP-adres, maar ze zijn niet het echte netwerk van de computer.

##Opslag
##Ik gebruik alleen de systeemschijf (C:). Die bestaat op elke Windows-computer, dus de gegevens zijn vergelijkbaar tussen alle groepsleden. Zo heeft elke computer ook maar één opslagwaarde, dus één rij in het CSV-bestand.

##Eenheden
##RAM en opslag staan in GB (met 2 decimalen), de CPU-frequentie staat in MHz. Alle groepsleden gebruiken dezelfde eenheden.

##Computermodel
##Het computermodel staat niet in het basisscript. Ik lees het uit met PowerShell (Win32_ComputerSystem).

import json
import ipaddress
import platform
import socket
import subprocess
import psutil
%run collect_sysinfo.ipynb

GB = 1024 ** 3
VIRTUAL_NAME_HINTS = ("virtual", "vmware", "vbox", "vethernet", "loopback", "bluetooth",
                      "vpn", "tap", "tun", "docker", "wsl", "pseudo", "hyper-v", "mcafee")
VIRTUAL_MAC_PREFIXES = ("0A-00-27", "00-05-9A", "00-50-56", "00-0C-29", "00-15-5D")


def run_powershell(cmd):
    try:
        out = subprocess.run(["powershell", "-NoProfile", "-Command", cmd],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        return json.loads(out) if out else {}
    except Exception:
        return {}


def get_model_and_serial():
    if platform.system() == "Windows":
        cs = run_powershell("Get-CimInstance Win32_ComputerSystem | Select-Object Manufacturer,Model | ConvertTo-Json")
        bio = run_powershell("Get-CimInstance Win32_BIOS | Select-Object SerialNumber | ConvertTo-Json")
        model = f"{cs.get('Manufacturer', '')} {cs.get('Model', '')}".strip() or None
        return model, bio.get("SerialNumber")

    def read(p):
        try:
            return open(p).read().strip()
        except Exception:
            return None
    vendor, name = read("/sys/class/dmi/id/sys_vendor"), read("/sys/class/dmi/id/product_name")
    return (f"{vendor or ''} {name or ''}".strip() or None), read("/sys/class/dmi/id/product_serial")


def is_primary_candidate(name, iface):
    if not iface["stats"] or not iface["stats"]["isup"]:
        return False
    if any(h in name.lower() for h in VIRTUAL_NAME_HINTS):
        return False
    ipv4 = [a for a in iface["addrs"] if a["family"] == socket.AF_INET]
    if not ipv4:
        return False
    ip = ipaddress.ip_address(ipv4[0]["address"])
    if ip.is_loopback or ip.is_link_local:
        return False
    mac = next((a["address"].upper().replace(":", "-") for a in iface["addrs"]
                if a["family"] == psutil.AF_LINK), "")
    return not mac.startswith(VIRTUAL_MAC_PREFIXES)


def find_primary_interface(info):
    ifaces = info["network"]["interfaces"]
    candidates = {n: i for n, i in ifaces.items() if is_primary_candidate(n, i)}
    if not candidates:
        return None
    return max(candidates, key=lambda n: (candidates[n]["io"] or {}).get("bytes_recv", 0))


def filter_info(info):
    model, serial = get_model_and_serial()
    freq = info["cpu"]["cpu_freq"]
    sys_mount = "C:\\" if platform.system() == "Windows" else "/"
    disk = info["storage"].get(sys_mount) or next(iter(info["storage"].values()), {})
    primary = find_primary_interface(info)
    ipv4 = mac = ipv6 = ipv6_ll = None
    if primary:
        for a in info["network"]["interfaces"][primary]["addrs"]:
            if a["family"] == psutil.AF_LINK:
                mac = a["address"]
            elif a["family"] == socket.AF_INET and ipv4 is None:
                ipv4 = a["address"]
            elif a["family"] == socket.AF_INET6:
                addr = a["address"].split("%")[0]
                if addr.lower().startswith("fe80"):
                    ipv6_ll = ipv6_ll or addr
                else:
                    ipv6 = ipv6 or addr
    return {
        "computernaam": info["network"]["hostname"],
        "computermodel": model,
        "serienummer": serial,                       
        "processor": info["cpu"]["processor"],
        "fysieke_cores": info["cpu"]["physical_cores"],
        "max_cpu_frequentie_mhz": round(freq["max"], 0) if freq else None,
        "ram_totaal_gb": round(info["memory"]["total"] / GB, 2),
        "opslag_bestandssysteem": disk.get("fstype"),
        "opslag_capaciteit_gb": round(disk.get("total", 0) / GB, 2),
        "primaire_interface": primary,
        "mac_adres": mac,
        "ipv4_adres": ipv4,
        "ipv6_adres": ipv6,
        "ipv6_link_local": ipv6_ll,
    }


def get_filtered():
    return filter_info(collect_system_info())


if __name__ == "__main__":
    print(json.dumps(get_filtered(), indent=2, ensure_ascii=False))

{
  "cpu": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "Intel64 Family 6 Model 154 Stepping 3, GenuineIntel",
    "physical_cores": 8,
    "logical_processors": 12,
    "cpu_freq": {
      "current": 2000.0,
      "min": 0.0,
      "max": 2000.0
    },
    "cpu_percent_total": 26.8,
    "cpu_percent_per_cpu": [
      14.3,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      16.7,
      42.9,
      14.3,
      28.6
    ]
  },
  "memory": {
    "total": 16866672640,
    "available": 3620237312,
    "percent": 78.5,
    "used": 13246435328,
    "free": 3620237312
  },
  "swap": {
    "total": 16866672640,
    "used": 1046376448,
    "free": 15820296192,
    "percent": 6.2,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 485510606848,
      "used": 443823169536,
      "free": 41687437312,
      "percent": 91.4
    }
  },
  "network": {
  